# MTA – Aufschreibung bereinigen Batch

Aktualisierte Version v10 als Ausgangslage, ergänzt um Störfall-Spalten in der zweiten Datei.


In [1]:
# MTA – Aufschreibung bereinigen
#
# Ausgangsbasis: Skript 1
# Ergänzungen:
# - Verarbeitung einer frei pflegbaren Dateiliste
# - Ergänzung auf Basis v10: zweite Datei enthält Störfall-Kennzeichen und Anzahl Störfälle je Zeitfenster
# - Auto-Erkennung der Kopfzeile für unterschiedlich aufgebaute Excel-Dateien
# - Unterstützung für Dateien mit "Datum" statt "DatumNEU"
# - Dauer Arbeitszeit auf 0 Nachkommastellen runden, z. B. 59.9999 -> 60.0
# - "Menge Gesamt (Stück)" in "MengeGesamtNIO" umbenennen
# - Zeilen löschen, wenn Bemerkung UND Station/OP leer sind
# - Mengen-Spalten N.i.O. / i.O. L4 / i.O. L5 von NULL auf 0 setzen
# - Dauer-/Ausfall-/Defizit-Spalten von NULL auf 0 setzen
# - Station/OP-Splitting aus Skript 1 bleibt erhalten
# - Ergänzung der Spalten Jahr, Monat, Tag, Quartal aus DatumNEU
# - Entfernen der Spalten Std. und Log
# - Schicht-Werte immer klein schreiben
# - DatumNEU als reines Datum ohne Uhrzeit speichern
# - Wochentag komplett neu aus DatumNEU/Datum berechnen
# - KW komplett neu aus DatumNEU/Datum berechnen
# - Zeilen ohne gültiges Datum werden entfernt
# - Anzahl MA NULL -> 0
# - Anzahl/ Std. und Sollzeit/ Stück (Min) NULL -> 0
# - Schicht wird in der zweiten Datei auch für ergänzte Zeilen aus der Ursprungsdatei übernommen
# - Zeilen mit Datum vor 2020 werden gelöscht
# - MengeGesamtNIO wird neu als Summe aus Menge N.i.O. + Menge i.O. L4 + Menge i.O. L5 berechnet
# - Dauer Arbeits-zeit wird neu als Differenz aus Zeit von und Zeit bis in Minuten berechnet
# - Zusätzlich wird eine zweite Datei erzeugt, die fehlende Datum/Zeit-Kombinationen aus der Ursprungsdatei ergänzt

import re
import unicodedata
import datetime
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from rapidfuzz import process, fuzz

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module="openpyxl"
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# =========================================================
# DATEIEN / PARAMETER
# =========================================================

SHEET_NAME = "Aufschreibung"

# Hier die zu verarbeitenden Dateien eintragen.
# Beispiel: weitere Dateien einfach als neue Path(...)-Zeile ergänzen.
DATEIEN_LISTE = [
    #Path(r"../data/raw/Störliste STW-Mittelteilanlage 2023_NEU.xlsx"),
     Path(r"../data/raw/mta2024to2026/Raw_unmerged/STW-Mittelteilanlage 2024_filled.xlsx"),
     Path(r"../data/raw/mta2024to2026/Raw_unmerged/Störliste STW-Mittelteilanlage 2025_filled.xlsx"),
     Path(r"../data/raw/mta2024to2026/Raw_unmerged/Störliste STW-Mittelteilanlage 2026_filled.xlsx"),
]

OUT_DIR = Path(r"../data/lstm ready data")
OUT_BASENAME = "aufschreibung_mta_clean_gesamt_v10"

# Zweiter Export:
# Diese Datei enthält alle Zeilen der ersten Ausgabedatei plus je eine zusätzliche Zeile
# für jede Datum/Zeit-von/Zeit-bis-Kombination aus den Ursprungsdateien, die in der
# ersten Ausgabedatei nicht mehr vorkommt.
CREATE_FILE_WITH_MISSING_TIME_COMBINATIONS = True
OUT_BASENAME_WITH_MISSING_INTERVALS = f"{OUT_BASENAME}_mit_fehlenden_zeitfenstern"

# Wie Werte aus der Ursprungsdatei für die zusätzlich erzeugten Zeilen je Zeitfenster
# übernommen werden sollen. "max" vermeidet Doppelzählungen bei mehrfach wiederholten
# Werten im gleichen Zeitfenster. Alternativen: "first" oder "sum".
MISSING_COMBINATION_VALUE_AGGREGATION = "max"

# Falls du nachvollziehen willst, aus welcher Datei eine Zeile kommt: True setzen.
ADD_SOURCE_FILE_COLUMN = False

# Kopfzeilen-Erkennung:
# - None = automatisch suchen
# - 0    = erste Excel-Zeile als Header verwenden
# - 1    = zweite Excel-Zeile als Header verwenden, usw.
HEADER_ROW = None

# Wenn HEADER_ROW = None ist, wird in den ersten Zeilen nach einer Zeile gesucht,
# die z. B. "Datum", "Schicht", "Zeit von", "Station/ OP" oder "Bemerkung" enthält.
AUTO_DETECT_HEADER_ROW = True

# Zeilen löschen, wenn Bemerkung UND Station/OP leer sind.
# Hinweis: Wenn du reine Produktionszeilen ohne Bemerkung/Station behalten möchtest,
# diesen Wert auf False setzen.
DROP_ROWS_IF_BEMERKUNG_AND_STATION_EMPTY = True

# Zeilen mit Datum vor diesem Jahr werden vollständig entfernt.
MIN_VALID_YEAR = 2020


# =========================================================
# 1) Spalten / Text Helpers
# =========================================================

def _normalize_colname(c: object) -> str:
    c = "" if c is None else str(c)
    c = unicodedata.normalize("NFKC", c)
    c = c.replace("\n", " ")
    c = re.sub(r"\s+", " ", c).strip()
    c = re.sub(r"[‐-‒–—―]", "-", c)
    c = re.sub(r"\s*/\s*", "/ ", c)
    c = re.sub(r"\s+", " ", c).strip()
    return c


_CANON_PATTERNS = [
    # Datum / Zeit
    (r"^datum\s*neu$", "DatumNEU"),
    (r"^datum$", "Datum"),
    (r"^wochen\s*tag$", "Wochentag"),
    (r"^wochentag$", "Wochentag"),
    (r"^schicht$", "Schicht"),
    (r"^zeit\s*von$", "Zeit von"),
    (r"^zeit\s*v(?:on)?\.?$", "Zeit von"),
    (r"^zeit\s*bis$", "Zeit bis"),
    (r"^zeit\s*b(?:is)?\.?$", "Zeit bis"),

    # Mengen / Dauer / Ausfall
    (r"^dauer\s+arbeit.*$", "Dauer Arbeits-zeit"),
    (r"^anzahl\s+ma$", "Anzahl MA"),
    (r"^anzahl\s*/\s*std\.?$", "Anzahl/ Std."),
    (r"^menge\s+n\.?\s*i\.?\s*o\.?$", "Menge N.i. O."),
    (r"^menge\s+i\.?\s*o\.?\s*l4$", "Menge i. O. L4"),
    (r"^menge\s+i\.?\s*o\.?\s*l5$", "Menge i. O. L5"),
    (r"^menge\s+gesamt.*$", "Menge Gesamt (Stück)"),
    (r"^sollzeit\s*/\s*stück.*$", "Sollzeit/ Stück (Min)"),
    (r"^dauer\s+org.*mang.*$", "Dauer Org-Mangel"),
    (r"^dauer\s+anlagen[-\s]*ausfall\s*intern.*$", "Dauer Anlagen-Ausfall intern"),
    (r"^dauer\s+anlagen[-\s]*ausfall.*$", "Dauer Anlagen-Ausfall"),
    (r"^störung\s+aufgrund\s+vormater.*$", "Störung aufgrund Vormaterial"),
    (r"^dauer\s+logistik.*defizit.*$", "Dauer Logistik- Defizite"),

    # Freitext / Station
    (r"^station\s*/\s*op$", "Station/ OP"),
    (r"^station/op$", "Station/ OP"),
    (r"^unterbrechungsursache$", "Unterbrechungsursache"),
    (r"^bemerkung$", "Bemerkung"),
]


def canonicalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    cols = [_normalize_colname(c) for c in df.columns]
    canon = []

    for c in cols:
        c2 = c
        for pat, repl in _CANON_PATTERNS:
            if re.match(pat, c2, flags=re.IGNORECASE):
                c2 = repl
                break
        canon.append(c2)

    # Doppelte Spaltennamen abfangen
    seen = {}
    out = []
    for c in canon:
        if c not in seen:
            seen[c] = 0
            out.append(c)
        else:
            seen[c] += 1
            out.append(f"{c}_{seen[c]}")

    df = df.copy()
    df.columns = out
    return df


def find_col(cols, patterns) -> str | None:
    """Wie in Skript 1: robustes Finden der Station/OP-Spalte inkl. Fallback."""
    for pat in patterns:
        for c in cols:
            if re.match(pat, c, flags=re.IGNORECASE):
                return c

    # fallback nur für Station/OP-Suche
    for c in cols:
        lc = c.lower()
        if "station" in lc and "op" in lc:
            return c
    return None


def find_col_by_patterns(cols, patterns) -> str | None:
    """Allgemeine Spaltensuche ohne Station/OP-Fallback."""
    for pat in patterns:
        for c in cols:
            if re.match(pat, c, flags=re.IGNORECASE):
                return c
    return None


def normalize_free_text(s: pd.Series) -> pd.Series:
    s = s.astype("string")
    s = s.map(lambda x: unicodedata.normalize("NFKC", x) if pd.notna(x) else x)
    s = s.str.lower()
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    s = s.str.replace(r"[‐-‒–—―]", "-", regex=True)
    s = s.str.replace(r"[•·●]", " ", regex=True)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    s = s.replace({"": pd.NA, "nan": pd.NA, "none": pd.NA, "k.a.": pd.NA, "k. a.": pd.NA})
    return s


def fuzzy_standardize(norm_s: pd.Series, threshold: int = 97, min_count: int = 2):
    # Vereinheitlicht NUR sehr ähnliche Schreibweisen (Typo/Spacing).
    counts = norm_s.dropna().value_counts()
    variants = counts.index.tolist()

    mapping = {}
    for v in variants:
        if v in mapping:
            continue
        mapping[v] = v
        matches = process.extract(
            v,
            variants,
            scorer=fuzz.token_sort_ratio,
            score_cutoff=threshold,
            limit=None
        )
        for m, score, _ in matches:
            if m not in mapping:
                mapping[m] = v

    std = norm_s.map(mapping).astype("string")
    std = std.where(norm_s.notna(), pd.NA)

    map_df = pd.DataFrame({
        "original": list(mapping.keys()),
        "standard": list(mapping.values()),
        "count": [counts.get(k, 0) for k in mapping.keys()]
    }).sort_values(["standard", "count"], ascending=[True, False])

    if min_count > 1:
        map_df = map_df[map_df["count"] >= min_count].copy()

    return std, map_df


# =========================================================
# 2) Excel-Lesen / Kopfzeile finden
# =========================================================

def _header_row_score(row: pd.Series) -> int:
    vals = [_normalize_colname(v) for v in row.tolist() if pd.notna(v)]
    text = " | ".join(vals).lower()
    if not text.strip():
        return 0

    checks = [
        r"\bdatum\b",
        r"wochen\s*tag",
        r"\bschicht\b",
        r"zeit\s*von",
        r"zeit\s*bis",
        r"dauer\s+arbeit",
        r"anzahl\s+ma",
        r"menge",
        r"station\s*/\s*op",
        r"bemerkung",
    ]
    return sum(1 for pat in checks if re.search(pat, text, flags=re.IGNORECASE))


def detect_header_row(path: Path, sheet_name: str | int | None, max_rows: int = 50) -> int:
    sheet = 0 if sheet_name in (None, "") else sheet_name
    preview = pd.read_excel(path, sheet_name=sheet, header=None, nrows=max_rows)

    best_idx = 0
    best_score = -1
    for idx, row in preview.iterrows():
        score = _header_row_score(row)
        if score > best_score:
            best_idx = int(idx)
            best_score = score

    # Falls keine plausible Kopfzeile gefunden wurde, normales Verhalten: erste Zeile.
    return best_idx if best_score >= 3 else 0


def read_aufschreibung_excel(path: Path) -> pd.DataFrame:
    sheet = 0 if SHEET_NAME in (None, "") else SHEET_NAME

    if HEADER_ROW is not None:
        header_row = HEADER_ROW
    elif AUTO_DETECT_HEADER_ROW:
        header_row = detect_header_row(path, sheet)
    else:
        header_row = 0

    df = pd.read_excel(path, sheet_name=sheet, header=header_row)
    df = df.dropna(axis=1, how="all")
    return df


# =========================================================
# 3) Datum / Zeit Helpers
# =========================================================

def _parse_date_series_to_datetime(s: pd.Series) -> pd.Series:
    """
    Parst Datumsspalten robust.

    Unterstützt u. a.:
    - echte Excel-/Datetime-Werte
    - deutsche Text-Daten wie 08.01.2024
    - Excel-Seriennummern wie 45299

    Wichtig: Reine Zahlen werden nur dann als Excel-Seriennummer interpretiert,
    wenn sie in einem plausiblen Datumsbereich liegen.
    """
    raw = s.copy()
    result = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    numeric = pd.to_numeric(raw, errors="coerce")
    excel_serial_mask = numeric.between(20000, 80000) & raw.notna()

    if excel_serial_mask.any():
        result.loc[excel_serial_mask] = pd.to_datetime(
            numeric.loc[excel_serial_mask],
            unit="D",
            origin="1899-12-30",
            errors="coerce"
        ).dt.normalize()

    other_mask = ~excel_serial_mask & raw.notna()
    if other_mask.any():
        result.loc[other_mask] = pd.to_datetime(
            raw.loc[other_mask],
            errors="coerce",
            dayfirst=True
        ).dt.normalize()

    return result


def parse_excel_date_to_datetime(s: pd.Series) -> pd.Series:
    return _parse_date_series_to_datetime(s)


def ensure_date_only_series(s: pd.Series) -> pd.Series:
    dt = _parse_date_series_to_datetime(s)
    dates = pd.Series(dt.dt.date, index=s.index, dtype="object")
    return dates.where(dt.notna(), pd.NA)


def parse_excel_time_to_str(s: pd.Series) -> pd.Series:
    def conv(x):
        if pd.isna(x):
            return pd.NA
        if isinstance(x, datetime.time):
            return x.strftime("%H:%M:%S")
        if isinstance(x, (pd.Timestamp, datetime.datetime)):
            return x.time().strftime("%H:%M:%S")
        if isinstance(x, (float, int, np.floating, np.integer)):
            # Excel-Uhrzeit als Tagesanteil: 0.5 = 12:00:00
            seconds = int(round(float(x) * 24 * 3600)) % (24 * 3600)
            h = seconds // 3600
            m = (seconds % 3600) // 60
            sec = seconds % 60
            return f"{h:02d}:{m:02d}:{sec:02d}"
        txt = str(x).strip()
        if not txt:
            return pd.NA
        t = pd.to_datetime(txt, errors="coerce")
        if pd.isna(t):
            return pd.NA
        return t.time().strftime("%H:%M:%S")

    return s.map(conv).astype("string")


def time_str_to_minutes(s: pd.Series) -> pd.Series:
    def conv(x):
        if pd.isna(x):
            return np.nan
        parts = str(x).split(":")
        if len(parts) < 2:
            return np.nan
        try:
            h = int(parts[0])
            m = int(parts[1])
            sec = int(parts[2]) if len(parts) > 2 else 0
            return h * 60 + m + sec / 60
        except ValueError:
            return np.nan

    return s.map(conv).astype(float)


def ensure_datumneu_column(df: pd.DataFrame) -> pd.DataFrame:
    """Erzeugt DatumNEU aus Datum, wenn eine Datei nur die Spalte Datum hat."""
    df = df.copy()
    datumneu_col = find_col_by_patterns(df.columns, [r"^DatumNEU$"])
    if datumneu_col:
        return df

    datum_col = find_col_by_patterns(df.columns, [r"^Datum$"])
    if not datum_col:
        return df

    insert_at = list(df.columns).index(datum_col) + 1
    df.insert(insert_at, "DatumNEU", df[datum_col])
    return df


def parse_date_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Parst Datum und DatumNEU robust.

    Wenn beide Spalten existieren, aber DatumNEU leer/ungültig ist,
    wird DatumNEU zeilenweise aus Datum ergänzt. Das verhindert, dass
    eine leere DatumNEU-Spalte gültige Datum-Werte aus der Spalte Datum überdeckt.
    """
    df = df.copy()
    for col in ["Datum", "DatumNEU"]:
        if col in df.columns:
            df[col] = parse_excel_date_to_datetime(df[col])

    if "Datum" in df.columns and "DatumNEU" in df.columns:
        df["DatumNEU"] = df["DatumNEU"].where(df["DatumNEU"].notna(), df["Datum"])

    return df


def get_primary_date_col(df: pd.DataFrame) -> str | None:
    """Bevorzugt DatumNEU, fällt sonst auf Datum zurück."""
    return find_col_by_patterns(df.columns, [r"^DatumNEU$"]) or find_col_by_patterns(df.columns, [r"^Datum$"])


def drop_rows_without_valid_date(df: pd.DataFrame) -> pd.DataFrame:
    """
    Entfernt alle Zeilen, in denen kein gültiges Datum ermittelt werden kann,
    sowie alle Zeilen mit Datum vor MIN_VALID_YEAR.
    """
    df = df.copy()
    date_col = get_primary_date_col(df)
    if not date_col:
        raise ValueError("Weder 'DatumNEU' noch 'Datum' gefunden. Ohne Datum kann keine Zeile verarbeitet werden.")

    dt = _parse_date_series_to_datetime(df[date_col])
    df[date_col] = dt

    min_date = pd.Timestamp(year=MIN_VALID_YEAR, month=1, day=1)
    keep = dt.notna() & (dt >= min_date)
    df = df.loc[keep].copy()

    return df


# =========================================================
# 4) Station/OP split Helpers aus Skript 1
# =========================================================

def _clean_station_token(token: object) -> str:
    t = unicodedata.normalize("NFKC", str(token))
    t = t.strip()
    t = re.sub(r"(?i)\b(R|OP)\.\b", r"\1 ", t)
    t = re.sub(r"(?i)\b(R|OP)\.", r"\1 ", t)
    t = re.sub(r"(?i)\b(R|OP)\s*([0-9])", r"\1 \2", t)
    t = re.sub(r"\s+", " ", t).strip()
    t = re.sub(r"(?i)^\s*op\b", "OP", t)
    t = re.sub(r"(?i)^\s*r\b", "R", t)
    return t


def split_station_op_simple(x: object) -> list[str]:
    if pd.isna(x):
        return []
    t = _clean_station_token(x)
    t = re.sub(r"[,/]", "|", t)
    parts = [_clean_station_token(p) for p in t.split("|")]
    return [p for p in parts if p and p.lower() not in ("nan", "none")]


def split_station_op_mta(x: object) -> list[str]:
    # MTA-Spezial:
    # - Trennung bei , oder /
    # - Zusätzlich: wenn in einem Chunk ein 2. 'R' oder 'OP' auftaucht, beginnt ein neuer Wert.
    if pd.isna(x):
        return []

    t = _clean_station_token(x)
    t = re.sub(r"[,/]", "|", t)
    chunks = [c.strip() for c in t.split("|") if c.strip()]

    out = []
    for ch in chunks:
        matches = list(re.finditer(r"(?i)\b(?:R|OP)\b", ch))
        if len(matches) <= 1:
            out.append(_clean_station_token(ch))
        else:
            pos = [m.start() for m in matches]
            for i, p in enumerate(pos):
                end = pos[i + 1] if i + 1 < len(pos) else len(ch)
                seg = ch[p:end].strip()
                if seg:
                    out.append(_clean_station_token(seg))

    seen = set()
    final = []
    for v in out:
        if v and v not in seen:
            seen.add(v)
            final.append(v)
    return final


def expand_split_columns(
    df: pd.DataFrame,
    source_col: str,
    splitter,
    prefix: str = "Station/ OP"
) -> pd.DataFrame:
    lists = df[source_col].map(splitter)
    max_len = int(lists.map(len).max()) if len(lists) else 0

    df2 = df.copy()
    df2[f"{prefix}_raw"] = df2[source_col].astype("string")

    for i in range(max_len):
        df2[f"{prefix}_{i + 1}"] = lists.map(
            lambda L: L[i] if len(L) > i else pd.NA
        ).astype("string")

    if max_len > 0:
        df2[source_col] = df2[f"{prefix}_1"]
    else:
        df2[source_col] = df2[source_col].astype("string")

    return df2


def drop_rows_empty_from(df: pd.DataFrame, start_col: str) -> tuple[pd.DataFrame, list[str]]:
    cols = list(df.columns)
    start_idx = cols.index(start_col)
    cols_from = cols[start_idx:]

    df2 = df.copy()
    for c in cols_from:
        if df2[c].dtype == object or str(df2[c].dtype).startswith("string"):
            df2[c] = df2[c].astype("string").str.strip()
            df2.loc[df2[c].isin(["", "nan", "NaN", "None", "<NA>"]), c] = pd.NA

    keep = df2[cols_from].notna().any(axis=1)
    return df2.loc[keep].copy(), cols_from


# =========================================================
# 5) Fachliche Bereinigungsregeln
# =========================================================

def _to_numeric_series(s: pd.Series) -> pd.Series:
    # Robust für Zahlen, die evtl. als Text mit deutschem Dezimalkomma vorliegen.
    if s.dtype == object or str(s.dtype).startswith("string"):
        s = s.astype("string").str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")


def _is_empty_value_series(s: pd.Series) -> pd.Series:
    txt = s.astype("string").str.strip()
    return s.isna() | txt.isna() | txt.isin(["", "nan", "NaN", "None", "<NA>", "<na>"])


def fill_nulls_with_zero(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    df = df.copy()
    for c in columns:
        if c in df.columns:
            df[c] = _to_numeric_series(df[c]).fillna(0.0)
    return df


def _get_single_col(df: pd.DataFrame, patterns: list[str]) -> str | None:
    """Hilfsfunktion: findet eine Spalte anhand mehrerer regulärer Ausdrücke."""
    return find_col_by_patterns(df.columns, patterns)


def _find_quantity_columns(df: pd.DataFrame) -> tuple[str | None, str | None, str | None]:
    """Findet die drei Mengen-Spalten robust, auch bei leicht abweichender Schreibweise."""
    menge_nio_col = _get_single_col(
        df,
        [r"^Menge\s+N\.?\s*i\.?\s*O\.?$", r"^Menge\s+N.*i.*O.*$"]
    )
    menge_l4_col = _get_single_col(
        df,
        [r"^Menge\s+i\.?\s*O\.?\s*L4$", r"^Menge.*i.*O.*L4$"]
    )
    menge_l5_col = _get_single_col(
        df,
        [r"^Menge\s+i\.?\s*O\.?\s*L5$", r"^Menge.*i.*O.*L5$"]
    )
    return menge_nio_col, menge_l4_col, menge_l5_col


def recalculate_menge_gesamt_nio(df: pd.DataFrame) -> pd.DataFrame:
    """
    Berechnet MengeGesamtNIO neu als Zeilensumme aus:
    Menge N.i. O. + Menge i. O. L4 + Menge i. O. L5.

    Nullwerte in den drei Einzelspalten werden vorher als 0 interpretiert.
    Die alte Quellspalte "Menge Gesamt (Stück)" wird nicht mehr übernommen,
    sondern durch die neu berechnete Summe ersetzt.
    """
    df = df.copy()

    menge_nio_col, menge_l4_col, menge_l5_col = _find_quantity_columns(df)
    summen_cols = [c for c in [menge_nio_col, menge_l4_col, menge_l5_col] if c]

    # Alte Gesamtspalte entfernen/umbenennen, damit es am Ende genau eine Zielspalte gibt.
    old_total_col = _get_single_col(
        df,
        [r"^Menge\s+Gesamt\s*\(\s*Stück\s*\)$", r"^Menge\s+Gesamt.*$"]
    )
    if old_total_col and old_total_col != "MengeGesamtNIO":
        if "MengeGesamtNIO" in df.columns:
            df = df.drop(columns=[old_total_col])
        else:
            df = df.rename(columns={old_total_col: "MengeGesamtNIO"})

    if not summen_cols:
        return df

    # Einzelmengen numerisch machen und NULL -> 0 setzen.
    for c in summen_cols:
        df[c] = _to_numeric_series(df[c]).fillna(0.0)

    df["MengeGesamtNIO"] = df[summen_cols].sum(axis=1).astype(float)

    # Zielspalte nach Möglichkeit direkt hinter der letzten Einzelmengen-Spalte platzieren.
    for ref_col in [menge_l5_col, menge_l4_col, menge_nio_col]:
        if ref_col in df.columns:
            df = _move_columns_after(df, ["MengeGesamtNIO"], ref_col)
            break

    return df


def recalculate_work_duration_minutes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Berechnet Dauer Arbeits-zeit neu als Differenz Zeit bis - Zeit von in Minuten.

    Beispiel: 04:45 bis 06:00 = 75.0.
    Falls Zeit bis kleiner als Zeit von ist, wird ein Tageswechsel angenommen
    und 24 Stunden addiert, z. B. 22:00 bis 06:00 = 480.0.
    """
    df = df.copy()

    t_from_col = _get_single_col(df, [r"^Zeit von$"])
    t_to_col = _get_single_col(df, [r"^Zeit bis$"])
    if not t_from_col or not t_to_col:
        return df

    # Sicherstellen, dass Minuten-Spalten existieren und aktuell sind.
    df[t_from_col] = parse_excel_time_to_str(df[t_from_col])
    df[t_to_col] = parse_excel_time_to_str(df[t_to_col])
    df["Zeit_von_min"] = time_str_to_minutes(df[t_from_col])
    df["Zeit_bis_min"] = time_str_to_minutes(df[t_to_col])

    duration = df["Zeit_bis_min"] - df["Zeit_von_min"]

    # Tageswechsel behandeln, aber nur bei gültigen Start-/Endzeiten.
    valid = df["Zeit_von_min"].notna() & df["Zeit_bis_min"].notna()
    duration = duration.where(~(valid & (duration < 0)), duration + 24 * 60)
    duration = duration.where(valid, np.nan).round(0).astype(float)

    arbeitszeit_col = _get_single_col(
        df,
        [r"^Dauer\s+Arbeits-?\s*zeit$", r"^Dauer\s+Arbeit.*$", r"^Dauer\s+Arbeits.*$"]
    )

    if arbeitszeit_col and arbeitszeit_col != "Dauer Arbeits-zeit":
        if "Dauer Arbeits-zeit" in df.columns:
            df = df.drop(columns=[arbeitszeit_col])
        else:
            df = df.rename(columns={arbeitszeit_col: "Dauer Arbeits-zeit"})

    df["Dauer Arbeits-zeit"] = duration

    # Zielspalte direkt hinter Zeit bis platzieren.
    if t_to_col in df.columns:
        df = _move_columns_after(df, ["Dauer Arbeits-zeit"], t_to_col)

    return df


def _move_columns_after(df: pd.DataFrame, columns_to_move: list[str], after_col: str) -> pd.DataFrame:
    """Verschiebt vorhandene Spalten direkt hinter eine Referenzspalte."""
    cols = [c for c in df.columns if c not in columns_to_move]
    if after_col not in cols:
        return df

    insert_at = cols.index(after_col) + 1
    for i, c in enumerate(columns_to_move):
        if c in df.columns:
            cols.insert(insert_at + i, c)
    return df.loc[:, cols]



def recalculate_weekday_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    Berechnet die Spalte Wochentag vollständig neu aus dem Datum.
    Logik: ISO-Wochentag als Zahl, also Montag=1, Dienstag=2, ..., Sonntag=7.
    Bevorzugt wird DatumNEU; falls nicht vorhanden, wird Datum verwendet.
    """
    df = df.copy()

    date_col = get_primary_date_col(df)
    if not date_col:
        return df

    dt = _parse_date_series_to_datetime(df[date_col])
    wochentag = (dt.dt.weekday + 1).astype("Int64")

    # Vorhandene Wochentag-Spalten bewusst entfernen, damit nichts Altes übernommen wird.
    existing_weekday_cols = [
        c for c in df.columns
        if re.match(r"^Wochentag(?:_\d+)?$", c, flags=re.IGNORECASE)
    ]
    if existing_weekday_cols:
        df = df.drop(columns=existing_weekday_cols)

    # Position wie in den Quelldateien: direkt hinter Datum, sonst hinter DatumNEU.
    insert_after = "Datum" if "Datum" in df.columns else date_col
    insert_at = list(df.columns).index(insert_after) + 1 if insert_after in df.columns else 0
    df.insert(insert_at, "Wochentag", wochentag)

    return df


def recalculate_kw_column(df: pd.DataFrame) -> pd.DataFrame:
    """
    Berechnet die Spalte KW vollständig neu aus dem Datum.
    Format wie im Ursprungsfile: ISO-Jahr/KW, z. B. 2024/02.
    """
    df = df.copy()

    date_col = get_primary_date_col(df)
    if not date_col:
        return df

    dt = _parse_date_series_to_datetime(df[date_col])
    iso = dt.dt.isocalendar()

    kw = pd.Series(pd.NA, index=df.index, dtype="string")
    valid = dt.notna()
    if valid.any():
        kw.loc[valid] = (
            iso.loc[valid, "year"].astype(str)
            + "/"
            + iso.loc[valid, "week"].astype(str).str.zfill(2)
        )

    # Vorhandene KW-Spalten bewusst entfernen, damit nichts Altes übernommen wird.
    existing_kw_cols = [
        c for c in df.columns
        if re.match(r"^KW(?:_\d+)?$", c, flags=re.IGNORECASE)
    ]
    if existing_kw_cols:
        df = df.drop(columns=existing_kw_cols)

    insert_after = "Wochentag" if "Wochentag" in df.columns else date_col
    insert_at = list(df.columns).index(insert_after) + 1 if insert_after in df.columns else 0
    df.insert(insert_at, "KW", kw)

    return df


def add_date_part_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Ergänzt Jahr, Monat, Tag und Quartal aus DatumNEU."""
    df = df.copy()
    date_col = find_col_by_patterns(df.columns, [r"^DatumNEU$"])
    if not date_col:
        return df

    dt = _parse_date_series_to_datetime(df[date_col])

    # Pandas Int64 erlaubt echte NULL-Werte bei Integer-Spalten.
    df["Jahr"] = dt.dt.year.astype("Int64")
    df["Monat"] = dt.dt.month.astype("Int64")
    df["Tag"] = dt.dt.day.astype("Int64")
    df["Quartal"] = dt.dt.quarter.astype("Int64")

    return _move_columns_after(df, ["Jahr", "Monat", "Tag", "Quartal"], date_col)


def ensure_selected_date_columns_are_date_only(df: pd.DataFrame) -> pd.DataFrame:
    """Stellt sicher, dass Datum und DatumNEU als reines Datum ohne Zeitanteil vorliegen."""
    df = df.copy()
    for col in ["Datum", "DatumNEU"]:
        if col in df.columns:
            df[col] = ensure_date_only_series(df[col])
    return df


def drop_unneeded_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Entfernt nicht mehr benötigte Spalten."""
    drop_cols = [c for c in ["Std.", "Log"] if c in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df


def normalize_shift_column(df: pd.DataFrame) -> pd.DataFrame:
    """Schreibt Schicht-Werte klein und behandelt Leerstrings als NULL."""
    df = df.copy()
    schicht_col = find_col_by_patterns(df.columns, [r"^Schicht$"])
    if not schicht_col:
        return df

    s_shift = df[schicht_col].astype("string").str.strip().str.lower()
    s_shift = s_shift.replace({"": pd.NA, "nan": pd.NA, "none": pd.NA, "<na>": pd.NA})
    df[schicht_col] = s_shift
    return df


def apply_custom_rules(
    df: pd.DataFrame,
    station_col: str,
    drop_empty_bemerkung_station_rows: bool | None = None
) -> pd.DataFrame:
    df = df.copy()

    if drop_empty_bemerkung_station_rows is None:
        drop_empty_bemerkung_station_rows = DROP_ROWS_IF_BEMERKUNG_AND_STATION_EMPTY

    # 1) Wochentag vollständig neu aus DatumNEU/Datum berechnen
    df = recalculate_weekday_column(df)

    # 2) KW vollständig neu aus DatumNEU/Datum berechnen
    df = recalculate_kw_column(df)

    # 3) Neue Datums-Spalten aus DatumNEU ergänzen
    df = add_date_part_columns(df)

    # Datum und DatumNEU als reines Datum ohne Uhrzeit speichern
    df = ensure_selected_date_columns_are_date_only(df)

    # 2) Nicht mehr benötigte Spalten entfernen
    df = drop_unneeded_columns(df)

    # 3) Schicht-Werte immer klein schreiben
    df = normalize_shift_column(df)

    # 4) Dauer Arbeits-zeit neu aus Zeit von / Zeit bis berechnen
    df = recalculate_work_duration_minutes(df)

    # 5) MengeGesamtNIO neu als Summe der drei Mengen-Spalten berechnen
    df = recalculate_menge_gesamt_nio(df)

    # 6) Anzahl MA von NULL auf 0 setzen
    anzahl_ma_col = find_col_by_patterns(
        df.columns,
        [r"^Anzahl\s+MA$", r"^Anzahl.*MA$"]
    )
    if anzahl_ma_col:
        df = fill_nulls_with_zero(df, [anzahl_ma_col])

    # 7) Anzahl/ Std. und Sollzeit/ Stück (Min) von NULL auf 0 setzen
    weitere_leistungs_cols = []
    for patterns in [
        [r"^Anzahl\s*/\s*Std\.?$", r"^Anzahl.*Std\.?$"],
        [r"^Sollzeit\s*/\s*Stück\s*\(\s*Min\s*\)$", r"^Sollzeit.*Stück.*Min.*$"],
    ]:
        c = find_col_by_patterns(df.columns, patterns)
        if c:
            weitere_leistungs_cols.append(c)
    df = fill_nulls_with_zero(df, weitere_leistungs_cols)

    # 8) Dauer Org-Mangel von NULL auf 0 setzen
    dauer_org_mangel_col = find_col_by_patterns(
        df.columns,
        [r"^Dauer\s+Org-?\s*Mangel$", r"^Dauer\s+Org.*$"]
    )
    if dauer_org_mangel_col:
        df = fill_nulls_with_zero(df, [dauer_org_mangel_col])

    # 9) Alle Dauer-Anlagen-Ausfall-Spalten von NULL auf 0 setzen
    # Enthält z. B. "Dauer Anlagen-Ausfall" und "Dauer Anlagen-Ausfall intern".
    dauer_anlagen_ausfall_cols = [
        c for c in df.columns
        if re.match(r"^Dauer\s+Anlagen[-\s]*Ausfall", c, flags=re.IGNORECASE)
    ]
    df = fill_nulls_with_zero(df, dauer_anlagen_ausfall_cols)

    # 10) Weitere Ausfall-/Defizit-Spalten von NULL auf 0 setzen
    weitere_null_zu_zero_cols = []
    for patterns in [
        [r"^Störung\s+aufgrund\s+Vormaterial$", r"^Störung\s+aufgrund\s+Vormater.*$"],
        [r"^Dauer\s+Logistik-?\s*Defizite$", r"^Dauer\s+Logistik.*Defizit.*$"],
    ]:
        c = find_col_by_patterns(df.columns, patterns)
        if c:
            weitere_null_zu_zero_cols.append(c)
    df = fill_nulls_with_zero(df, weitere_null_zu_zero_cols)

    # 11) Zeilen löschen, wenn Bemerkung UND Station/OP leer sind
    if drop_empty_bemerkung_station_rows:
        bemerkung_col = find_col_by_patterns(df.columns, [r"^Bemerkung$"])
        if bemerkung_col:
            drop_mask = _is_empty_value_series(df[bemerkung_col]) & _is_empty_value_series(df[station_col])
            df = df.loc[~drop_mask].copy()

    return df




def _time_combination_key_frame(df: pd.DataFrame) -> tuple[pd.DataFrame | None, pd.Series | None]:
    """Erzeugt einen normierten Schlüssel aus Datum, Zeit von und Zeit bis."""
    date_col = get_primary_date_col(df)
    t_from_col = _get_single_col(df, [r"^Zeit von$"])
    t_to_col = _get_single_col(df, [r"^Zeit bis$"])

    if not date_col or not t_from_col or not t_to_col:
        return None, None

    key = pd.DataFrame(index=df.index)
    key["_key_datum"] = _parse_date_series_to_datetime(df[date_col]).dt.date
    key["_key_zeit_von"] = parse_excel_time_to_str(df[t_from_col])
    key["_key_zeit_bis"] = parse_excel_time_to_str(df[t_to_col])

    valid = (
        key["_key_datum"].notna()
        & key["_key_zeit_von"].notna()
        & key["_key_zeit_bis"].notna()
    )
    return key, valid


def _key_tuples(key_df: pd.DataFrame) -> set[tuple[object, object, object]]:
    return set(
        key_df[["_key_datum", "_key_zeit_von", "_key_zeit_bis"]]
        .itertuples(index=False, name=None)
    )


def _aggregate_numeric_for_missing_row(s: pd.Series, mode: str | None = None) -> float:
    """Aggregiert Werte aus der Ursprungsdatei für eine zusätzliche Zeitfenster-Zeile."""
    mode = (mode or MISSING_COMBINATION_VALUE_AGGREGATION).lower()
    vals = _to_numeric_series(s).dropna()
    if vals.empty:
        return 0.0
    if mode == "sum":
        return float(vals.sum())
    if mode == "first":
        return float(vals.iloc[0])
    # Standard: max. Verhindert Doppelzählungen, falls dieselbe Stundenleistung
    # im Ursprungsblatt in mehreren Zeilen des gleichen Zeitfensters wiederholt wird.
    return float(vals.max())


def _aggregate_text_for_missing_row(s: pd.Series) -> object:
    """Übernimmt den ersten nicht-leeren Textwert aus der Ursprungsdatei."""
    vals = (
        s.astype("string")
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "none": pd.NA, "<na>": pd.NA})
        .dropna()
    )
    if vals.empty:
        return pd.NA
    return vals.iloc[0]


def _duration_minutes_from_time_strings(time_from: object, time_to: object) -> float:
    start = time_str_to_minutes(pd.Series([time_from])).iloc[0]
    end = time_str_to_minutes(pd.Series([time_to])).iloc[0]
    if pd.isna(start) or pd.isna(end):
        return np.nan
    duration = float(end - start)
    if duration < 0:
        duration += 24 * 60
    return float(round(duration, 0))


def _placeholder_should_be_zero(col: str) -> bool:
    """Spalten, die in ergänzten Zeitfenster-Zeilen standardmäßig 0 erhalten sollen."""
    zero_patterns = [
        r"^Anzahl\b",
        r"^Menge\b",
        r"^MengeGesamtNIO$",
        r"^Dauer\b",
        r"^Störung\s+aufgrund\s+Vormaterial$",
        r"^Takt\b",
        r"^Produktionszeit\b",
        r"^Sollzeit\b",
        r"^Unnamed",
    ]
    return any(re.match(pat, col, flags=re.IGNORECASE) for pat in zero_patterns)



def add_stoerfall_columns_for_second_export(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ergänzt in der zweiten Ausgabedatei:
    - Störfall = J für Datensätze aus der ersten Datei
    - Störfall = N für ergänzte Zeitfenster-Zeilen ohne Störung
    - Anzahl Störfälle Zeitfenster = Anzahl J-Zeilen je DatumNEU/Zeit von/Zeit bis
    """
    df_out = df.copy()

    if "Störfall" not in df_out.columns:
        df_out["Störfall"] = "J"
    else:
        df_out["Störfall"] = (
            df_out["Störfall"]
            .astype("string")
            .str.strip()
            .str.upper()
        )
        df_out["Störfall"] = df_out["Störfall"].fillna("J")
        df_out.loc[~df_out["Störfall"].isin(["J", "N"]), "Störfall"] = "J"

    key, valid = _time_combination_key_frame(df_out)
    count_col = "Anzahl Störfälle Zeitfenster"

    if key is None or valid is None:
        df_out[count_col] = 0
        return df_out

    key_cols = ["_key_datum", "_key_zeit_von", "_key_zeit_bis"]
    tmp = key.copy()
    tmp["_row_order"] = range(len(tmp))
    tmp["_stoerfall"] = df_out["Störfall"].to_numpy()
    tmp["_valid"] = valid.to_numpy()

    counts = (
        tmp.loc[tmp["_valid"] & tmp["_stoerfall"].eq("J"), key_cols]
        .groupby(key_cols, dropna=False)
        .size()
        .rename(count_col)
        .reset_index()
    )

    tmp = tmp.merge(counts, how="left", on=key_cols)
    tmp = tmp.sort_values("_row_order")
    df_out[count_col] = tmp[count_col].fillna(0).astype(int).to_numpy()

    return df_out

def build_dataset_with_missing_time_combinations(
    df_first_output: pd.DataFrame,
    df_original_reference: pd.DataFrame
) -> pd.DataFrame:
    """
    Erzeugt die zweite Ausgabedatei.

    Inhalt:
    - alle Zeilen der ersten Ausgabedatei
    - plus je eine zusätzliche Zeile für jede Kombination aus DatumNEU/Datum,
      Zeit von und Zeit bis, die in den Ursprungsdaten vorkommt, aber in der
      ersten Ausgabedatei fehlt.

    Für die zusätzlichen Zeilen werden die Datums-/Zeitspalten neu berechnet.
    Schicht, Anzahl MA und Mengen werden aus der Ursprungsdatei übernommen bzw. je
    Zeitfenster aggregiert. Fehlende Werte werden auf 0 gesetzt.

    Zusätzlich wird ausschließlich in dieser zweiten Datei gekennzeichnet:
    - Störfall = J für Zeilen aus der ersten Ausgabedatei
    - Störfall = N für ergänzte Zeitfenster-Zeilen ohne Störung
    - Anzahl Störfälle Zeitfenster = Anzahl J-Zeilen je DatumNEU/Zeit von/Zeit bis
    """
    df_first = df_first_output.copy()
    df_first["Störfall"] = "J"
    df_ref = df_original_reference.copy()

    first_key, first_valid = _time_combination_key_frame(df_first)
    ref_key, ref_valid = _time_combination_key_frame(df_ref)

    if first_key is None or first_valid is None or ref_key is None or ref_valid is None:
        print("Zweiter Export übersprungen: Datum/Zeit-von/Zeit-bis-Spalten nicht vollständig vorhanden.")
        return add_stoerfall_columns_for_second_export(df_first)

    first_keys = _key_tuples(first_key.loc[first_valid].drop_duplicates())

    df_ref_valid = df_ref.loc[ref_valid].copy()
    df_ref_valid[["_key_datum", "_key_zeit_von", "_key_zeit_bis"]] = ref_key.loc[
        ref_valid,
        ["_key_datum", "_key_zeit_von", "_key_zeit_bis"]
    ]

    ref_unique_keys = (
        df_ref_valid[["_key_datum", "_key_zeit_von", "_key_zeit_bis"]]
        .drop_duplicates()
    )
    missing_keys = [
        tuple(row)
        for row in ref_unique_keys.itertuples(index=False, name=None)
        if tuple(row) not in first_keys
    ]

    if not missing_keys:
        print("Zweiter Export: Keine fehlenden Datum/Zeit-Kombinationen gefunden.")
        return add_stoerfall_columns_for_second_export(df_first)

    missing_key_set = set(missing_keys)
    key_tuple_series = list(
        df_ref_valid[["_key_datum", "_key_zeit_von", "_key_zeit_bis"]]
        .itertuples(index=False, name=None)
    )
    df_ref_missing = df_ref_valid.loc[[k in missing_key_set for k in key_tuple_series]].copy()

    rows = []
    quantity_cols = [c for c in _find_quantity_columns(df_ref_missing) if c]
    anzahl_ma_col = find_col_by_patterns(df_ref_missing.columns, [r"^Anzahl\s+MA$", r"^Anzahl.*MA$"])
    schicht_col = find_col_by_patterns(df_ref_missing.columns, [r"^Schicht$"])

    for (datum, zeit_von, zeit_bis), group in df_ref_missing.groupby(
        ["_key_datum", "_key_zeit_von", "_key_zeit_bis"],
        sort=True,
        dropna=False
    ):
        row = {col: pd.NA for col in df_first.columns}
        if "Störfall" in row:
            row["Störfall"] = "N"

        # Datum / Zeit aus dem Schlüssel setzen
        if "Datum" in row:
            row["Datum"] = datum
        if "DatumNEU" in row:
            row["DatumNEU"] = datum
        if "Zeit von" in row:
            row["Zeit von"] = zeit_von
        if "Zeit bis" in row:
            row["Zeit bis"] = zeit_bis

        # Datumswerte trotzdem berechnen
        dt = pd.Timestamp(datum)
        if "Wochentag" in row:
            row["Wochentag"] = int(dt.weekday() + 1)
        if "KW" in row:
            iso = dt.isocalendar()
            row["KW"] = f"{iso.year}/{iso.week:02d}"
        if "Jahr" in row:
            row["Jahr"] = int(dt.year)
        if "Monat" in row:
            row["Monat"] = int(dt.month)
        if "Tag" in row:
            row["Tag"] = int(dt.day)
        if "Quartal" in row:
            row["Quartal"] = int(dt.quarter)

        # Zeitwerte / Dauer Arbeitszeit berechnen
        zeit_von_min = time_str_to_minutes(pd.Series([zeit_von])).iloc[0]
        zeit_bis_min = time_str_to_minutes(pd.Series([zeit_bis])).iloc[0]
        if "Zeit_von_min" in row:
            row["Zeit_von_min"] = zeit_von_min
        if "Zeit_bis_min" in row:
            row["Zeit_bis_min"] = zeit_bis_min
        if "Dauer Arbeits-zeit" in row:
            row["Dauer Arbeits-zeit"] = _duration_minutes_from_time_strings(zeit_von, zeit_bis)

        # Standardmäßig NULL/0-Werte für ergänzte Zeilen
        for col in df_first.columns:
            if _placeholder_should_be_zero(col) and pd.isna(row[col]):
                row[col] = 0.0

        # Schicht aus Ursprungsdatei übernehmen. In der Referenz ist sie bereits klein geschrieben.
        if schicht_col and schicht_col in row and schicht_col in group.columns:
            row[schicht_col] = _aggregate_text_for_missing_row(group[schicht_col])

        # Anzahl MA aus Ursprungsdatei übernehmen/aggregieren
        if anzahl_ma_col and anzahl_ma_col in row and anzahl_ma_col in group.columns:
            row[anzahl_ma_col] = _aggregate_numeric_for_missing_row(group[anzahl_ma_col])

        # Mengen aus Ursprungsdatei übernehmen/aggregieren
        for q_col in quantity_cols:
            if q_col in row and q_col in group.columns:
                row[q_col] = _aggregate_numeric_for_missing_row(group[q_col])

        # MengeGesamtNIO aus der zusätzlichen Zeile neu berechnen
        if "MengeGesamtNIO" in row:
            row["MengeGesamtNIO"] = float(
                sum(float(row.get(c, 0.0) or 0.0) for c in quantity_cols if c in row)
            )

        # Quelle_Datei optional aus erster Ursprungszeile übernehmen
        if "Quelle_Datei" in row and "Quelle_Datei" in group.columns:
            non_null_sources = group["Quelle_Datei"].dropna()
            row["Quelle_Datei"] = non_null_sources.iloc[0] if len(non_null_sources) else pd.NA

        rows.append(row)

    df_missing_rows = pd.DataFrame(rows, columns=df_first.columns)
    df_out = pd.concat([df_first, df_missing_rows], axis=0, ignore_index=True)

    # Sortierung: Datum, Zeit von, Zeit bis. Danach Datum wieder als reines Date speichern.
    date_col = get_primary_date_col(df_out)
    if date_col:
        df_out["_sort_datum"] = _parse_date_series_to_datetime(df_out[date_col])
    if "Zeit_von_min" in df_out.columns:
        df_out["_sort_zeit_von"] = _to_numeric_series(df_out["Zeit_von_min"])
    elif "Zeit von" in df_out.columns:
        df_out["_sort_zeit_von"] = time_str_to_minutes(parse_excel_time_to_str(df_out["Zeit von"]))
    if "Zeit_bis_min" in df_out.columns:
        df_out["_sort_zeit_bis"] = _to_numeric_series(df_out["Zeit_bis_min"])
    elif "Zeit bis" in df_out.columns:
        df_out["_sort_zeit_bis"] = time_str_to_minutes(parse_excel_time_to_str(df_out["Zeit bis"]))

    sort_cols = [c for c in ["_sort_datum", "_sort_zeit_von", "_sort_zeit_bis"] if c in df_out.columns]
    if sort_cols:
        df_out = df_out.sort_values(by=sort_cols).reset_index(drop=True)
        df_out = df_out.drop(columns=sort_cols)

    df_out = ensure_selected_date_columns_are_date_only(df_out)
    df_out = add_stoerfall_columns_for_second_export(df_out)
    print(f"Zweiter Export: {len(df_missing_rows)} fehlende Datum/Zeit-Kombinationen ergänzt.")
    return df_out


def export_dataframe(df: pd.DataFrame, basename: str) -> tuple[Path, Path]:
    """Exportiert CSV und Excel mit reinem Datumsformat."""
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    out_csv = OUT_DIR / f"{basename}.csv"
    out_xlsx = OUT_DIR / f"{basename}.xlsx"

    df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    with pd.ExcelWriter(out_xlsx, engine="openpyxl", date_format="yyyy-mm-dd", datetime_format="yyyy-mm-dd") as writer:
        df.to_excel(writer, index=False)

    return out_csv, out_xlsx


# =========================================================
# 6) Einzeldatei bereinigen
# =========================================================

def bereinige_datensatz(
    df_raw: pd.DataFrame,
    source_file: Path | None = None,
    return_unfiltered: bool = False
) -> pd.DataFrame | tuple[pd.DataFrame, pd.DataFrame]:
    df = canonicalize_columns(df_raw)
    df = ensure_datumneu_column(df)

    # Spalten robust finden
    station_col = find_col(df.columns, [r"^Station/\s*OP$"])
    if station_col is None:
        raise ValueError(
            "Spalte 'Station/ OP' nicht gefunden. Bitte Spaltennamen prüfen. "
            "Bei Dateien mit anderer Kopfzeilenposition ggf. HEADER_ROW setzen."
        )

    # Wenn der Fallback eine anders geschriebene Station/OP-Spalte gefunden hat, standardisieren.
    if station_col != "Station/ OP":
        df = df.rename(columns={station_col: "Station/ OP"})
        station_col = "Station/ OP"

    t_from_col = find_col_by_patterns(df.columns, [r"^Zeit von$"])
    t_to_col = find_col_by_patterns(df.columns, [r"^Zeit bis$"])

    # Datum zuerst robust parsen und Zeilen ohne gültiges Datum entfernen.
    df = parse_date_columns(df)
    df = drop_rows_without_valid_date(df)

    # Keine pauschale Löschung ab Station/OP mehr an dieser Stelle.
    # Die gewünschte Regel "Bemerkung UND Station/OP leer" wird später gezielt
    # in apply_custom_rules() angewendet. So können Dauer/Mengen vorher neu berechnet werden.

    if t_from_col:
        df[t_from_col] = parse_excel_time_to_str(df[t_from_col])
        df["Zeit_von_min"] = time_str_to_minutes(df[t_from_col])

    if t_to_col:
        df[t_to_col] = parse_excel_time_to_str(df[t_to_col])
        df["Zeit_bis_min"] = time_str_to_minutes(df[t_to_col])

    # Station/OP aufspalten (MTA-Regeln aus Skript 1)
    df = expand_split_columns(
        df,
        source_col=station_col,
        splitter=split_station_op_mta,
        prefix="Station/ OP"
    )

    # Neue Regeln anwenden, zunächst OHNE den Bemerkung/Station-Filter.
    # Diese ungefilterte Referenz wird für den zweiten Export benötigt.
    df_unfiltered = apply_custom_rules(
        df,
        station_col=station_col,
        drop_empty_bemerkung_station_rows=False
    )

    # Erste Ausgabedatei: wie bisher mit gewünschtem Filter.
    if DROP_ROWS_IF_BEMERKUNG_AND_STATION_EMPTY:
        bemerkung_col = find_col_by_patterns(df_unfiltered.columns, [r"^Bemerkung$"])
        if bemerkung_col:
            drop_mask = _is_empty_value_series(df_unfiltered[bemerkung_col]) & _is_empty_value_series(df_unfiltered[station_col])
            df_bereinigt = df_unfiltered.loc[~drop_mask].copy()
        else:
            df_bereinigt = df_unfiltered.copy()
    else:
        df_bereinigt = df_unfiltered.copy()

    if ADD_SOURCE_FILE_COLUMN and source_file is not None:
        df_bereinigt["Quelle_Datei"] = source_file.name
        df_unfiltered["Quelle_Datei"] = source_file.name

    if return_unfiltered:
        return df_bereinigt, df_unfiltered

    return df_bereinigt


# =========================================================
# 7) Alle Dateien verarbeiten + zusammenführen
# =========================================================

def main() -> pd.DataFrame:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    bereinigte_datenframes = []
    referenz_datenframes = []

    for datei in DATEIEN_LISTE:
        print(f"Verarbeite Datei: {datei}")
        df_raw = read_aufschreibung_excel(datei)
        df_bereinigt, df_referenz = bereinige_datensatz(df_raw, source_file=datei, return_unfiltered=True)
        bereinigte_datenframes.append(df_bereinigt)
        referenz_datenframes.append(df_referenz)

    if not bereinigte_datenframes:
        raise ValueError("DATEIEN_LISTE ist leer. Bitte mindestens eine Datei eintragen.")

    df_gesamt = pd.concat(
        bereinigte_datenframes,
        axis=0,
        ignore_index=True
    )

    df_referenz_gesamt = pd.concat(
        referenz_datenframes,
        axis=0,
        ignore_index=True
    )

    # Optional sortieren. Dafür temporär als datetime interpretieren, danach wieder als reines Datum speichern.
    if "DatumNEU" in df_gesamt.columns:
        df_gesamt["_sort_DatumNEU"] = _parse_date_series_to_datetime(df_gesamt["DatumNEU"])
        df_gesamt = (
            df_gesamt
            .sort_values(by="_sort_DatumNEU")
            .drop(columns=["_sort_DatumNEU"])
            .reset_index(drop=True)
        )
        df_gesamt = ensure_selected_date_columns_are_date_only(df_gesamt)

    # Freitext vereinheitlichen auf dem gesamten kombinierten Datensatz.
    # Dadurch entstehen konsistente Standards über alle Dateien hinweg.
    for free_col in ["Bemerkung", "Unterbrechungsursache"]:
        if free_col in df_gesamt.columns:
            df_gesamt[f"{free_col}_norm"] = normalize_free_text(df_gesamt[free_col])
            df_gesamt[f"{free_col}_std"], map_df = fuzzy_standardize(
                df_gesamt[f"{free_col}_norm"],
                threshold=97,
                min_count=2
            )

            safe = re.sub(r"[^a-z0-9]+", "_", free_col.lower())
            mapping_path = OUT_DIR / f"mta_mapping_{safe}.xlsx"
            map_df.to_excel(mapping_path, index=False)
            print(f"Mapping gespeichert: {mapping_path}")

    print(df_gesamt.head(10))
    print("Bereinigt gesamt:", df_gesamt.shape)

    # =========================================================
    # 8) Export
    # =========================================================

    OUT_CSV, OUT_XLSX = export_dataframe(df_gesamt, OUT_BASENAME)
    print("Gespeichert:", OUT_CSV, "und", OUT_XLSX)

    if CREATE_FILE_WITH_MISSING_TIME_COMBINATIONS:
        df_mit_fehlenden = build_dataset_with_missing_time_combinations(
            df_first_output=df_gesamt,
            df_original_reference=df_referenz_gesamt
        )
        OUT_CSV_2, OUT_XLSX_2 = export_dataframe(
            df_mit_fehlenden,
            OUT_BASENAME_WITH_MISSING_INTERVALS
        )
        print("Gespeichert zweite Datei:", OUT_CSV_2, "und", OUT_XLSX_2)

    return df_gesamt


if __name__ == "__main__":
    main()


Verarbeite Datei: ..\data\raw\mta2024to2026\Raw_unmerged\STW-Mittelteilanlage 2024_filled.xlsx
Verarbeite Datei: ..\data\raw\mta2024to2026\Raw_unmerged\Störliste STW-Mittelteilanlage 2025_filled.xlsx
Verarbeite Datei: ..\data\raw\mta2024to2026\Raw_unmerged\Störliste STW-Mittelteilanlage 2026_filled.xlsx
Mapping gespeichert: ..\data\lstm ready data\mta_mapping_bemerkung.xlsx
        Datum  Wochentag       KW    DatumNEU  Jahr  Monat  Tag  Quartal Schicht  Zeit von  Zeit bis  Dauer Arbeits-zeit  Anzahl MA  Menge N.i. O.  Menge i. O. L4  Menge i. O. L5  MengeGesamtNIO  \
0  2024-07-10          5  2023/16  2023-04-21  2023      4   21        2       f  08:00:00  09:00:00                60.0        5.0            0.0             4.0             0.0             4.0   
1  2024-07-10          5  2023/16  2023-04-21  2023      4   21        2       f  08:00:00  09:00:00                60.0        5.0            0.0             4.0             0.0             4.0   
2  2024-01-08          1  202